# V18-v2 IT→IA Transition Gene Verification

**Purpose:** Verify Result 4.1 IT→IA transition genes using v2 tissue-separated data

**Issues to resolve:**
1. SERPINE1 — claimed as NK ↑ in Results, but C5-v1 shows Myeloid ↓. Which is correct in v2?
2. TGFB1, ID3 in Blood NK — IT→IA direction and p-value?
3. TOX, GZMK in Liver Myeloid — IT→IA direction and p-value?
4. Broader: What are ALL significant IT→IA transition genes (tissue-separated)?

**Method:** Donor-level Mann-Whitney U test, IT vs IA, separate for Liver and Blood

In [ ]:
!pip install scanpy anndata matplotlib seaborn scipy -q

# Cell 1: Mount Drive and load data
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'

print('Loading h5ad (backed mode)...')
adata = sc.read_h5ad(DATA_PATH, backed='r')
print(f'Shape: {adata.shape}')
print(f'Columns: {list(adata.obs.columns[:20])}')

In [ ]:
# Cell 2: Identify key columns
obs = adata.obs.copy()

# Stage/Disease column
stage_candidates = ['Stage', 'stage', 'disease', 'Disease', 'condition', 'group']
COL_STAGE = None
for c in stage_candidates:
    if c in obs.columns:
        COL_STAGE = c
        break
print(f'Stage column: {COL_STAGE}')
print(f'Stage values: {obs[COL_STAGE].value_counts().to_dict()}')

# Lineage column
lineage_candidates = ['major_lineage', 'lineage', 'cell_type', 'celltype']
COL_LINEAGE = None
for c in lineage_candidates:
    if c in obs.columns:
        COL_LINEAGE = c
        break
print(f'\nLineage column: {COL_LINEAGE}')
print(f'Lineage values: {obs[COL_LINEAGE].value_counts().to_dict()}')

# Tissue column
tissue_candidates = ['tissue', 'Tissue', 'compartment', 'source']
COL_TISSUE = None
for c in tissue_candidates:
    if c in obs.columns:
        COL_TISSUE = c
        break
# If no tissue column, try to derive from sample name
if COL_TISSUE is None:
    sample_candidates = ['sample', 'Sample', 'orig.ident', 'batch']
    for c in sample_candidates:
        if c in obs.columns:
            vals = obs[c].unique()[:5]
            print(f'\nSample column candidate: {c}, examples: {vals}')
            # Check if contains Liver/PBMC or L/P
            if any('Liver' in str(v) or 'PBMC' in str(v) or '_L_' in str(v) or '_P_' in str(v) for v in vals):
                print('  → Contains tissue info, deriving...')
                # Derive tissue
                obs['tissue_derived'] = obs[c].apply(
                    lambda x: 'Liver' if ('Liver' in str(x) or '_L_' in str(x) or str(x).endswith('_L'))
                    else ('Blood' if ('PBMC' in str(x) or '_P_' in str(x) or str(x).endswith('_P') or 'Blood' in str(x))
                    else 'Unknown'))
                COL_TISSUE = 'tissue_derived'
                break

if COL_TISSUE:
    print(f'\nTissue column: {COL_TISSUE}')
    print(f'Tissue values: {obs[COL_TISSUE].value_counts().to_dict()}')
else:
    print('\n⚠️ No tissue column found! Check obs columns manually:')
    print(list(obs.columns))

# Donor column
donor_candidates = ['donor', 'Donor', 'patient', 'subject', 'donor_id']
COL_DONOR = None
for c in donor_candidates:
    if c in obs.columns:
        COL_DONOR = c
        break
if COL_DONOR is None:
    # Derive from sample column: split by '_' and take [1]
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['donor_derived'] = obs[c].astype(str).str.split('_').str[1]
            COL_DONOR = 'donor_derived'
            break

print(f'\nDonor column: {COL_DONOR}')
n_donors = obs[COL_DONOR].nunique()
print(f'Unique donors: {n_donors}')
print(f'Donors per stage:')
for stage in sorted(obs[COL_STAGE].unique()):
    donors = obs[obs[COL_STAGE] == stage][COL_DONOR].unique()
    print(f'  {stage}: n={len(donors)} — {sorted(donors)}')

Stage column: Stage
Stage values: {'IA': 62545, 'IT': 49179, 'AR': 45452, 'CR': 43245, 'NL': 42579}

Lineage column: major_lineage
Lineage values: {'CD8_T': 70065, 'CD4_T': 69446, 'NK': 54584, 'Myeloid': 24716, 'B': 21325, 'PlasmaB': 2083, 'gdT': 781}

Tissue column: tissue
Tissue values: {'Blood': 136408, 'Liver': 106592}

Donor column: donor_derived
Unique donors: 23
Donors per stage:
  AR: n=3 — ['P190716', 'P191008', 'P191217']
  CR: n=3 — ['P191126', 'P191127', 'P191210']
  IA: n=5 — ['P190719', 'P190801', 'P190911', 'P191028', 'P191112']
  IT: n=6 — ['P190326', 'P190402', 'P190604', 'P190808', 'P190902', 'P190910']
  NL: n=6 — ['D528848', 'D529074', 'D529351', 'D529354', 'D529409', 'Dhc570']


In [ ]:
# Cell 3: Check which target genes exist in the dataset
gene_names = list(adata.var_names)

# Priority targets for Result 4.1
targets_41 = ['TGFB1', 'ID3', 'SERPINE1', 'TOX', 'GZMK', 'SNAI1',
               'KLRD1', 'PRDM1',  # NK IT→IA from C5
               'LAG3', 'NCAM1', 'ZEB1', 'CDK2', 'PFKM', 'SLC2A1', 'CDH1',  # Myeloid IT→IA from C5
               'FOXP3', 'GNLY', 'LAYN', 'TIGIT', 'DNMT1']  # 4.2 IA vs AR genes

found = [g for g in targets_41 if g in gene_names]
missing = [g for g in targets_41 if g not in gene_names]

print(f'Found: {len(found)}/{len(targets_41)}')
print(f'Found genes: {found}')
if missing:
    print(f'Missing: {missing}')

Found: 20/20
Found genes: ['TGFB1', 'ID3', 'SERPINE1', 'TOX', 'GZMK', 'SNAI1', 'KLRD1', 'PRDM1', 'LAG3', 'NCAM1', 'ZEB1', 'CDK2', 'PFKM', 'SLC2A1', 'CDH1', 'FOXP3', 'GNLY', 'LAYN', 'TIGIT', 'DNMT1']


In [ ]:
# ============================================================
# Cell 3B: DIAGNOSTIC — paste this BETWEEN Cell 3 and Cell 4
# Identifies the backed mode issue and loads data properly
# ============================================================

import scipy.sparse as sp

print('=== DIAGNOSTIC: Data access mode ===')
print(f'adata.isbacked: {adata.isbacked}')
print(f'adata.X type: {type(adata.X)}')

# Test direct access
test_gene = 'TGFB1'
test_idx = list(adata.var_names).index(test_gene)

# Test 1: Row slice
print('\n--- Test 1: Row slice adata.X[:10, :] ---')
try:
    chunk = adata.X[:10, :]
    print(f'  Type: {type(chunk)}, shape: {chunk.shape}')
    if hasattr(chunk, 'toarray'):
        vals = chunk[:, test_idx].toarray().flatten()
    else:
        vals = np.asarray(chunk[:, test_idx]).flatten()
    print(f'  Values: {vals}')
    print(f'  Any NaN: {np.any(np.isnan(vals))}')
except Exception as e:
    print(f'  FAILED: {e}')

# Test 2: Gene slice
print('\n--- Test 2: Gene slice adata[:, test_gene].X ---')
try:
    gene_data = adata[:, test_gene].X
    print(f'  Type: {type(gene_data)}, shape: {gene_data.shape}')
    if hasattr(gene_data, 'toarray'):
        vals = gene_data.toarray().flatten()[:10]
    else:
        vals = np.asarray(gene_data).flatten()[:10]
    print(f'  First 10: {vals}')
    print(f'  Any NaN: {np.any(np.isnan(vals))}')
except Exception as e:
    print(f'  FAILED: {e}')

# Test 3: Integer index access
print('\n--- Test 3: adata.X[0:10, test_idx] ---')
try:
    chunk = adata.X[0:10, test_idx]
    if hasattr(chunk, 'toarray'):
        vals = chunk.toarray().flatten()
    elif sp.issparse(chunk):
        vals = np.asarray(chunk.todense()).flatten()
    else:
        vals = np.asarray(chunk).flatten()
    print(f'  Values: {vals}')
except Exception as e:
    print(f'  FAILED: {e}')

# ============================================================
# If all tests fail or return NaN → reload WITHOUT backed mode
# ============================================================
print('\n--- Decision ---')

reload_needed = False
try:
    chunk = adata.X[:10, :]
    if hasattr(chunk, 'toarray'):
        test_vals = chunk[:, test_idx].toarray().flatten()
    else:
        test_vals = np.asarray(chunk[:, test_idx]).flatten()
    if np.all(np.isnan(test_vals)) or np.all(test_vals == 0):
        reload_needed = True
        print('⚠️ Values are all NaN or zero → backed mode not working properly')
    else:
        print(f'✅ Backed mode works. Sample values: {test_vals[:5]}')
except:
    reload_needed = True
    print('⚠️ Backed access failed entirely')

if reload_needed:
    print('\n🔄 Reloading WITHOUT backed mode (this may take a few minutes)...')
    adata = sc.read_h5ad(DATA_PATH)
    obs = adata.obs.copy()

    # Re-derive donor column if needed
    if COL_DONOR == 'donor_derived':
        for c in ['sample', 'Sample', 'orig.ident']:
            if c in obs.columns:
                obs['donor_derived'] = obs[c].astype(str).str.split('_').str[1]
                break
    if COL_TISSUE == 'tissue_derived':
        for c in ['sample', 'Sample', 'orig.ident']:
            if c in obs.columns:
                obs['tissue_derived'] = obs[c].apply(
                    lambda x: 'Liver' if ('Liver' in str(x) or '_L_' in str(x) or str(x).endswith('_L'))
                    else ('Blood' if ('PBMC' in str(x) or '_P_' in str(x) or str(x).endswith('_P') or 'Blood' in str(x))
                    else 'Unknown'))
                break

    # Verify
    chunk = adata.X[:10, :]
    if hasattr(chunk, 'toarray'):
        test_vals = chunk[:, test_idx].toarray().flatten()
    else:
        test_vals = np.asarray(chunk[:, test_idx]).flatten()
    print(f'✅ Reload complete. Shape: {adata.shape}')
    print(f'   Sample {test_gene} values: {test_vals[:5]}')
else:
    print('✅ No reload needed, backed mode is functional.')


=== DIAGNOSTIC: Data access mode ===
adata.isbacked: True
adata.X type: <class 'h5py._hl.dataset.Dataset'>

--- Test 1: Row slice adata.X[:10, :] ---
  Type: <class 'numpy.ndarray'>, shape: (10, 24452)
  Values: [0.        0.        1.2648576 2.297829  0.        1.5645908 0.9399366
 0.        1.5787961 1.493319 ]
  Any NaN: False

--- Test 2: Gene slice adata[:, test_gene].X ---
  Type: <class 'numpy.ndarray'>, shape: (243000, 1)
  First 10: [0.        0.        1.2648576 2.297829  0.        1.5645908 0.9399366
 0.        1.5787961 1.493319 ]
  Any NaN: False

--- Test 3: adata.X[0:10, test_idx] ---
  Values: [0.        0.        1.2648576 2.297829  0.        1.5645908 0.9399366
 0.        1.5787961 1.493319 ]

--- Decision ---
✅ Backed mode works. Sample values: [0.        0.        1.2648576 2.297829  0.       ]
✅ No reload needed, backed mode is functional.


In [ ]:
# ============================================================
# Cell 4 REPLACEMENT — paste this over the existing Cell 4
# Fix: backed='r' mode expression extraction
# ============================================================

def donor_level_test(adata, obs_df, gene, lineage, tissue,
                     group_a='IT', group_b='IA',
                     col_stage=COL_STAGE, col_lineage=COL_LINEAGE,
                     col_tissue=COL_TISSUE, col_donor=COL_DONOR):
    """
    Compute donor-level mean expression and Mann-Whitney U test.
    Fixed for backed='r' h5ad access.
    """
    # Filter cells
    mask = (
        (obs_df[col_stage].isin([group_a, group_b])) &
        (obs_df[col_lineage] == lineage) &
        (obs_df[col_tissue] == tissue)
    )
    cells = obs_df[mask].copy()

    if len(cells) == 0:
        return None

    # Get gene index
    gene_idx = list(adata.var_names).index(gene)

    # ===== FIX: Extract expression properly for backed mode =====
    cell_indices = np.where(mask.values)[0]

    # Method 1: slice adata first, then access X
    try:
        chunk = adata.X[cell_indices, :]
        if hasattr(chunk, 'toarray'):
            expr = np.asarray(chunk[:, gene_idx].toarray()).flatten()
        else:
            expr = np.asarray(chunk[:, gene_idx]).flatten()
    except Exception as e1:
        # Method 2: read gene column for ALL cells, then filter
        try:
            col = adata[:, gene].X
            if hasattr(col, 'toarray'):
                col = col.toarray().flatten()
            else:
                col = np.asarray(col).flatten()
            expr = col[cell_indices]
        except Exception as e2:
            print(f'  ⚠️ Expression extraction failed for {gene}/{lineage}/{tissue}: {e1} | {e2}')
            return None

    # Check for valid expression
    if len(expr) == 0 or np.all(np.isnan(expr)):
        return None

    cells['expr'] = expr

    # Donor-level means
    donor_means = cells.groupby([col_donor, col_stage])['expr'].mean().reset_index()

    vals_a = donor_means[donor_means[col_stage] == group_a]['expr'].dropna().values
    vals_b = donor_means[donor_means[col_stage] == group_b]['expr'].dropna().values

    n_a = len(vals_a)
    n_b = len(vals_b)

    if n_a < 2 or n_b < 2:
        return None

    mean_a = float(np.mean(vals_a))
    mean_b = float(np.mean(vals_b))

    # Mann-Whitney U test
    try:
        stat, p_val = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        p_val = float(p_val)
    except:
        p_val = 1.0

    # Percent change: group_b vs group_a
    if mean_a > 1e-10:
        pct_change = (mean_b - mean_a) / mean_a * 100
    else:
        pct_change = float('inf') if mean_b > 1e-10 else 0.0

    # Donor-pair consistency
    direction = 'up' if mean_b > mean_a else 'down'
    n_consistent = 0
    n_total = n_a * n_b
    for va in vals_a:
        for vb in vals_b:
            if direction == 'up' and vb > va:
                n_consistent += 1
            elif direction == 'down' and vb < va:
                n_consistent += 1

    return {
        'gene': gene,
        'lineage': lineage,
        'tissue': tissue,
        'comparison': f'{group_a}→{group_b}',
        f'n_{group_a}': n_a,
        f'n_{group_b}': n_b,
        f'mean_{group_a}': round(mean_a, 4),
        f'mean_{group_b}': round(mean_b, 4),
        'pct_change': round(pct_change, 1),
        'direction': '↑' if mean_b > mean_a else '↓',
        'p_value': round(p_val, 4),
        'sig': '★' if p_val < 0.05 else ('†' if p_val < 0.10 else ''),
        'consistency': f'{n_consistent}/{n_total}',
        f'donors_{group_a}': [round(float(v), 4) for v in sorted(vals_a)],
        f'donors_{group_b}': [round(float(v), 4) for v in sorted(vals_b)],
    }

# Quick test — verify extraction works
test_gene = 'TGFB1'
test_idx = list(adata.var_names).index(test_gene)
try:
    # Test Method 1
    test_chunk = adata.X[:100, :]
    if hasattr(test_chunk, 'toarray'):
        test_val = test_chunk[:, test_idx].toarray().flatten()
    else:
        test_val = np.asarray(test_chunk[:, test_idx]).flatten()
    print(f'Method 1 OK: {test_gene} first 100 cells, mean={np.mean(test_val):.4f}, non-zero={np.count_nonzero(test_val)}')
except Exception as e:
    print(f'Method 1 failed: {e}')
    # Test Method 2
    try:
        test_col = adata[:, test_gene].X
        if hasattr(test_col, 'toarray'):
            test_col = test_col.toarray().flatten()
        else:
            test_col = np.asarray(test_col).flatten()
        print(f'Method 2 OK: {test_gene} all cells, mean={np.mean(test_col):.4f}, non-zero={np.count_nonzero(test_col)}')
    except Exception as e2:
        print(f'Method 2 also failed: {e2}')
        print('→ Try loading WITHOUT backed mode: adata = sc.read_h5ad(DATA_PATH)')

print('\ndonor_level_test function defined ✅')


Method 1 OK: TGFB1 first 100 cells, mean=0.9086, non-zero=52

donor_level_test function defined ✅


In [ ]:
# Cell 5: Test priority targets for Result 4.1
# IT→IA transition genes (tissue-separated)

# Define targets: (gene, lineage, tissue)
priority_targets = [
    # Current Results 4.1 claims
    ('TGFB1', 'NK', 'Blood'),       # claimed NK ↑
    ('SERPINE1', 'NK', 'Blood'),     # claimed NK ↑ — SUSPECT
    ('ID3', 'NK', 'Blood'),          # claimed NK ↑
    ('TOX', 'Myeloid', 'Liver'),     # claimed Myeloid ↓
    ('GZMK', 'Myeloid', 'Liver'),    # claimed Myeloid ↓

    # Cross-check: SERPINE1 in Myeloid (C5-v1 showed Myeloid ↓)
    ('SERPINE1', 'Myeloid', 'Blood'),
    ('SERPINE1', 'Myeloid', 'Liver'),

    # Additional: check TGFB1/TOX/GZMK in both tissues
    ('TGFB1', 'NK', 'Liver'),
    ('TOX', 'Myeloid', 'Blood'),
    ('GZMK', 'Myeloid', 'Blood'),
    ('GZMK', 'NK', 'Blood'),
    ('GZMK', 'NK', 'Liver'),
    ('GZMK', 'CD8_T', 'Liver'),
    ('GZMK', 'CD8_T', 'Blood'),
]

# Also check IT→IA for each target in NL→IT for context
results = []

# Get lineage name mapping (check actual values)
actual_lineages = sorted(obs[COL_LINEAGE].unique())
print(f'Actual lineage names: {actual_lineages}\n')

# Build lineage name map
lineage_map = {}
for lin in actual_lineages:
    lineage_map[lin] = lin
# Common aliases
alias = {'NK': 'NK', 'Myeloid': 'Myeloid', 'CD4_T': 'CD4_T', 'CD8_T': 'CD8_T',
         'B': 'B', 'PlasmaB': 'PlasmaB'}
for k, v in alias.items():
    if v not in actual_lineages:
        # Try to find matching lineage
        for al in actual_lineages:
            if k.lower() in al.lower() or al.lower() in k.lower():
                alias[k] = al
                break
print(f'Lineage alias map: {alias}\n')

# Get tissue values
actual_tissues = sorted(obs[COL_TISSUE].unique())
print(f'Actual tissue names: {actual_tissues}\n')

# Tissue alias
tissue_alias = {'Liver': 'Liver', 'Blood': 'Blood'}
for t in actual_tissues:
    if 'liver' in t.lower():
        tissue_alias['Liver'] = t
    elif 'blood' in t.lower() or 'pbmc' in t.lower():
        tissue_alias['Blood'] = t
print(f'Tissue alias map: {tissue_alias}\n')

# Get stage values
actual_stages = sorted(obs[COL_STAGE].unique())
print(f'Actual stage names: {actual_stages}\n')

# Stage alias
stage_alias = {'IT': 'IT', 'IA': 'IA', 'NL': 'NL'}
for s in actual_stages:
    sl = s.lower().strip()
    if sl in ['it', 'immune_tolerant', 'immunotolerant']:
        stage_alias['IT'] = s
    elif sl in ['ia', 'immune_active', 'immuneactive']:
        stage_alias['IA'] = s
    elif sl in ['nl', 'normal', 'healthy']:
        stage_alias['NL'] = s
print(f'Stage alias map: {stage_alias}')

Actual lineage names: ['B', 'CD4_T', 'CD8_T', 'Myeloid', 'NK', 'PlasmaB', 'gdT']

Lineage alias map: {'NK': 'NK', 'Myeloid': 'Myeloid', 'CD4_T': 'CD4_T', 'CD8_T': 'CD8_T', 'B': 'B', 'PlasmaB': 'PlasmaB'}

Actual tissue names: ['Blood', 'Liver']

Tissue alias map: {'Liver': 'Liver', 'Blood': 'Blood'}

Actual stage names: ['AR', 'CR', 'IA', 'IT', 'NL']

Stage alias map: {'IT': 'IT', 'IA': 'IA', 'NL': 'NL'}


In [ ]:
# Cell 6: Run IT→IA tests on all priority targets

print('='*90)
print('IT→IA TRANSITION: PRIORITY TARGETS (Tissue-Separated, Donor-Level)')
print('='*90)

it_ia_results = []
nl_it_results = []  # for context

for gene, lin, tis in priority_targets:
    if gene not in adata.var_names:
        print(f'  ⚠️ {gene} not in dataset — SKIP')
        continue

    # Resolve aliases
    lin_actual = alias.get(lin, lin)
    tis_actual = tissue_alias.get(tis, tis)
    it_actual = stage_alias.get('IT', 'IT')
    ia_actual = stage_alias.get('IA', 'IA')
    nl_actual = stage_alias.get('NL', 'NL')

    # IT→IA test
    res = donor_level_test(adata, obs, gene, lin_actual, tis_actual,
                           group_a=it_actual, group_b=ia_actual)
    if res:
        it_ia_results.append(res)
        sig = res['sig']
        print(f'{sig:>1s} {gene:>10s} | {lin:>8s} | {tis:>6s} | '
              f'IT={res[f"mean_{it_actual}"]:.4f} → IA={res[f"mean_{ia_actual}"]:.4f} | '
              f'{res["direction"]}{abs(res["pct_change"]):.1f}% | '
              f'p={res["p_value"]:.4f} | {res["consistency"]}')
    else:
        print(f'  {gene:>10s} | {lin:>8s} | {tis:>6s} | NO DATA or insufficient donors')

    # Also NL→IT for context
    res_nl = donor_level_test(adata, obs, gene, lin_actual, tis_actual,
                              group_a=nl_actual, group_b=it_actual)
    if res_nl:
        nl_it_results.append(res_nl)

print(f'\nTotal IT→IA tests completed: {len(it_ia_results)}')

IT→IA TRANSITION: PRIORITY TARGETS (Tissue-Separated, Donor-Level)
★      TGFB1 |       NK |  Blood | IT=1.4759 → IA=2.1233 | ↑43.9% | p=0.0159 | 20/20
★   SERPINE1 |       NK |  Blood | IT=0.0027 → IA=0.0119 | ↑341.4% | p=0.0159 | 20/20
★        ID3 |       NK |  Blood | IT=0.0044 → IA=0.0380 | ↑770.3% | p=0.0317 | 19/20
★        TOX |  Myeloid |  Liver | IT=0.0860 → IA=0.0033 | ↓96.1% | p=0.0067 | 30/30
★       GZMK |  Myeloid |  Liver | IT=0.0957 → IA=0.0016 | ↓98.4% | p=0.0116 | 29/30
    SERPINE1 |  Myeloid |  Blood | IT=0.0032 → IA=0.0052 | ↑60.9% | p=0.6228 | 7/20
    SERPINE1 |  Myeloid |  Liver | IT=0.0306 → IA=0.0169 | ↓44.6% | p=0.3703 | 16/30
       TGFB1 |       NK |  Liver | IT=1.8899 → IA=2.0792 | ↑10.0% | p=0.5368 | 19/30
         TOX |  Myeloid |  Blood | IT=0.0034 → IA=0.0062 | ↑83.7% | p=0.2857 | 15/20
        GZMK |  Myeloid |  Blood | IT=0.0028 → IA=0.0081 | ↑192.6% | p=0.2683 | 15/20
        GZMK |       NK |  Blood | IT=0.2940 → IA=0.7683 | ↑161.3% | p=0.1111 | 1

In [ ]:
# Cell 7: Summary table — IT→IA with NL→IT context side by side

print('='*120)
print('COMPREHENSIVE COMPARISON: NL→IT vs IT→IA (Tissue-Separated, Donor-Level)')
print('='*120)
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL→IT":>20s} | {"IT→IA":>20s} | {"Pattern":>15s}')
print('-'*120)

it_actual = stage_alias.get('IT', 'IT')
ia_actual = stage_alias.get('IA', 'IA')
nl_actual = stage_alias.get('NL', 'NL')

for r_ia in it_ia_results:
    gene = r_ia['gene']
    lin = r_ia['lineage']
    tis = r_ia['tissue']

    # Find matching NL→IT
    r_nl = None
    for rn in nl_it_results:
        if rn['gene'] == gene and rn['lineage'] == lin and rn['tissue'] == tis:
            r_nl = rn
            break

    nl_str = '—'
    if r_nl:
        nl_str = f'{r_nl["sig"]}{r_nl["direction"]}{abs(r_nl["pct_change"]):.0f}% p={r_nl["p_value"]:.3f}'

    ia_str = f'{r_ia["sig"]}{r_ia["direction"]}{abs(r_ia["pct_change"]):.0f}% p={r_ia["p_value"]:.3f}'

    # Classify pattern
    is_nl_it_sig = r_nl and r_nl['p_value'] < 0.05
    is_it_ia_sig = r_ia['p_value'] < 0.05

    if is_nl_it_sig and is_it_ia_sig:
        pattern = 'TRANSITION'
    elif is_nl_it_sig and not is_it_ia_sig:
        pattern = 'IT-persistent'
    elif not is_nl_it_sig and is_it_ia_sig:
        pattern = 'IA-emergent'
    else:
        pattern = 'NS both'

    print(f'{gene:>10s} | {lin:>8s} | {tis:>6s} | '
          f'{nl_str:>20s} | {ia_str:>20s} | {pattern:>15s}')

COMPREHENSIVE COMPARISON: NL→IT vs IT→IA (Tissue-Separated, Donor-Level)
      Gene |  Lineage | Tissue |                NL→IT |                IT→IA |         Pattern
------------------------------------------------------------------------------------------------------------------------
     TGFB1 |       NK |  Blood |         ↑23% p=0.309 |        ★↑44% p=0.016 |     IA-emergent
  SERPINE1 |       NK |  Blood |         ↑88% p=0.204 |       ★↑341% p=0.016 |     IA-emergent
       ID3 |       NK |  Blood |        ↑111% p=0.290 |       ★↑770% p=0.032 |     IA-emergent
       TOX |  Myeloid |  Liver |      ★↑1857% p=0.004 |        ★↓96% p=0.007 |      TRANSITION
      GZMK |  Myeloid |  Liver |        ↑248% p=0.295 |        ★↓98% p=0.012 |     IA-emergent
  SERPINE1 |  Myeloid |  Blood |      ★↑1240% p=0.045 |         ↑61% p=0.623 |   IT-persistent
  SERPINE1 |  Myeloid |  Liver |         ↓25% p=0.684 |         ↓45% p=0.370 |         NS both
     TGFB1 |       NK |  Liver |          ↑4% 

In [ ]:
# Cell 8: BROAD SCAN — All C5 148 genes, IT→IA, tissue-separated
# Find ALL significant IT→IA changes to identify the complete transition signature

import os

# Load C5 gene list
c5_path = os.path.join(RESULTS_DIR, 'C5')
if not os.path.exists(c5_path):
    # Try alternate locations
    alt_paths = [
        '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/C5',
        '/content/drive/MyDrive/ITLAS/results/version18-analysis/C5',
    ]
    for ap in alt_paths:
        if os.path.exists(ap):
            c5_path = ap
            break

print(f'C5 path: {c5_path}')
if os.path.exists(c5_path):
    c5_files = os.listdir(c5_path)
    print(f'Files: {c5_files[:20]}')
    # Look for gene list CSV
    gene_list_files = [f for f in c5_files if 'gene' in f.lower() and f.endswith('.csv')]
    print(f'Gene list files: {gene_list_files}')
else:
    print('C5 directory not found')

# Also check for the authoritative gene list
gene_list_paths = [
    os.path.join(RESULTS_DIR, 'C4_selected_genes_for_C5.csv'),
    '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/C4_selected_genes_for_C5.csv',
    '/content/drive/MyDrive/ITLAS/results/version18-analysis/C4_selected_genes_for_C5.csv',
]
for gp in gene_list_paths:
    if os.path.exists(gp):
        c5_genes_df = pd.read_csv(gp)
        print(f'\nC5 gene list from {gp}')
        print(f'Shape: {c5_genes_df.shape}')
        print(f'Columns: {list(c5_genes_df.columns)}')
        print(f'First 5: {c5_genes_df.head()}')
        break
else:
    print('\n⚠️ C5 gene list not found. Using the 21 target genes from C8 transition analysis instead.')

C5 path: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/C5
Files: ['C5_genes_liver.csv', 'C5_genes_blood.csv']
Gene list files: ['C5_genes_liver.csv', 'C5_genes_blood.csv']

⚠️ C5 gene list not found. Using the 21 target genes from C8 transition analysis instead.


In [ ]:
# Cell 9: Focused scan — IT→IA for transition-relevant genes across key lineages
# Test genes most likely to show IT→IA transition:
# exhaustion markers, suppressive markers, metabolic, signaling

transition_genes = [
    # IT→IA candidates from C5-v1 and current Results
    'TGFB1', 'ID3', 'SERPINE1', 'TOX', 'GZMK', 'SNAI1',
    # IT-specific genes (should revert at IA)
    'SOCS1', 'SOCS3', 'LAYN', 'TOX2',
    # Suppressive markers
    'LGALS9', 'AIM2', 'MEFV', 'DNMT1', 'DNMT3A', 'TET2',
    # Key signaling
    'JAK1', 'MTOR', 'PRDM1',
    # IA-associated
    'FOXP3', 'GNLY', 'TIGIT', 'CTLA4', 'BCL6',
]

# Filter to genes that exist
transition_genes = [g for g in transition_genes if g in adata.var_names]
print(f'Testing {len(transition_genes)} genes across key lineages...\n')

# Key lineages for IT→IA
key_lineage_tissue = [
    ('NK', 'Blood'), ('NK', 'Liver'),
    ('Myeloid', 'Blood'), ('Myeloid', 'Liver'),
    ('CD4_T', 'Blood'), ('CD4_T', 'Liver'),
    ('CD8_T', 'Blood'), ('CD8_T', 'Liver'),
]

broad_results = []

for gene in transition_genes:
    for lin, tis in key_lineage_tissue:
        lin_actual = alias.get(lin, lin)
        tis_actual = tissue_alias.get(tis, tis)
        it_actual = stage_alias.get('IT', 'IT')
        ia_actual = stage_alias.get('IA', 'IA')

        res = donor_level_test(adata, obs, gene, lin_actual, tis_actual,
                               group_a=it_actual, group_b=ia_actual)
        if res and res['p_value'] < 0.10:  # include trends
            broad_results.append(res)

print(f'Total significant/trend IT→IA results: {len(broad_results)}\n')
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | {"Dir":>3s} | {"% Change":>10s} | {"p":>8s} | {"Cons":>8s}')
print('-'*75)

for r in sorted(broad_results, key=lambda x: x['p_value']):
    print(f'{r["sig"]:>1s}{r["gene"]:>9s} | {r["lineage"]:>8s} | {r["tissue"]:>6s} | '
          f'{r["direction"]:>3s} | {r["pct_change"]:>9.1f}% | '
          f'{r["p_value"]:>8.4f} | {r["consistency"]:>8s}')

Testing 24 genes across key lineages...

Total significant/trend IT→IA results: 12

      Gene |  Lineage | Tissue | Dir |   % Change |        p |     Cons
---------------------------------------------------------------------------
★      TOX |  Myeloid |  Liver |   ↓ |     -96.1% |   0.0067 |    30/30
★     GZMK |  Myeloid |  Liver |   ↓ |     -98.4% |   0.0116 |    29/30
★    TGFB1 |       NK |  Blood |   ↑ |      43.9% |   0.0159 |    20/20
★ SERPINE1 |       NK |  Blood |   ↑ |     341.4% |   0.0159 |    20/20
★     GZMK |    CD8_T |  Liver |   ↓ |     -22.7% |   0.0303 |    27/30
★      ID3 |       NK |  Blood |   ↑ |     770.3% |   0.0317 |    19/20
★ SERPINE1 |    CD4_T |  Liver |   ↑ |     534.1% |   0.0444 |    24/30
†   LGALS9 |       NK |  Liver |   ↓ |     -31.6% |   0.0519 |    26/30
†     MTOR |  Myeloid |  Liver |   ↓ |     -64.7% |   0.0617 |    24/30
†     AIM2 |    CD8_T |  Blood |   ↑ |     180.7% |   0.0635 |    18/20
†    CTLA4 |  Myeloid |  Liver |   ↓ |     -89.4

In [ ]:
# Cell 10: FINAL SUMMARY for manuscript revision

print('='*80)
print('FINAL VERIFICATION SUMMARY FOR RESULT 4.1')
print('='*80)

print()
print('--- CURRENT RESULTS 4.1 CLAIMS ---')
print()
print('1. SERPINE1 Blood NK ↑  →  Actual: [CHECK]')
print('2. TGFB1   Blood NK ↑  →  Actual: [CHECK]')
print('3. ID3     Blood NK ↑  →  Actual: [CHECK]')
print('4. TOX     Liver Myeloid ↓  →  Actual: [CHECK]')
print('5. GZMK    Liver Myeloid ↓  →  Actual: [CHECK]')
print()
print('--- ACTION ITEMS ---')
print('□ If SERPINE1 is NOT significant in Blood NK: remove from Results 4.1')
print('□ If SERPINE1 is significant in Myeloid instead: correct the lineage')
print('□ Add actual p-values and % changes to all IT→IA claims')
print('□ If TOX/GZMK in Liver Myeloid lack statistical support: revise claim')
print('□ Consider adding Supplementary Table for IT→IA transition data')
print()
print('--- SAVE RESULTS ---')

# Save all results to CSV
if it_ia_results:
    df_out = pd.DataFrame(it_ia_results)
    out_path = os.path.join(RESULTS_DIR, 'C8_IT_IA_transition_verification.csv')
    try:
        df_out.to_csv(out_path, index=False)
        print(f'Saved priority results to: {out_path}')
    except:
        out_path = '/content/drive/MyDrive/ITLAS/C8_IT_IA_transition_verification.csv'
        df_out.to_csv(out_path, index=False)
        print(f'Saved priority results to: {out_path}')

if broad_results:
    df_broad = pd.DataFrame(broad_results)
    out_path2 = os.path.join(RESULTS_DIR, 'C8_IT_IA_broad_scan.csv')
    try:
        df_broad.to_csv(out_path2, index=False)
        print(f'Saved broad scan to: {out_path2}')
    except:
        out_path2 = '/content/drive/MyDrive/ITLAS/C8_IT_IA_broad_scan.csv'
        df_broad.to_csv(out_path2, index=False)
        print(f'Saved broad scan to: {out_path2}')

print('\n✅ Verification complete. Review results before updating manuscript.')

FINAL VERIFICATION SUMMARY FOR RESULT 4.1

--- CURRENT RESULTS 4.1 CLAIMS ---

1. SERPINE1 Blood NK ↑  →  Actual: [CHECK]
2. TGFB1   Blood NK ↑  →  Actual: [CHECK]
3. ID3     Blood NK ↑  →  Actual: [CHECK]
4. TOX     Liver Myeloid ↓  →  Actual: [CHECK]
5. GZMK    Liver Myeloid ↓  →  Actual: [CHECK]

--- ACTION ITEMS ---
□ If SERPINE1 is NOT significant in Blood NK: remove from Results 4.1
□ If SERPINE1 is significant in Myeloid instead: correct the lineage
□ Add actual p-values and % changes to all IT→IA claims
□ If TOX/GZMK in Liver Myeloid lack statistical support: revise claim
□ Consider adding Supplementary Table for IT→IA transition data

--- SAVE RESULTS ---
Saved priority results to: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/C8_IT_IA_transition_verification.csv
Saved broad scan to: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/C8_IT_IA_broad_scan.csv

✅ Verification complete. Review results before updating manuscript.
